# Этап 1 — проверка подготовки данных в Google Colab

**Этот файл поставляется без результатов выполнения.** `pytest passed` и `complete`
появятся только после соответствующих фактических операций в Colab.
CPU достаточно. Перед подключением выберите Runtime → Change runtime type →
Runtime Version `2026.07` (Python 3.12); текущий default с Python 3.13 не подходит
этим pinned-зависимостям. См. [официальный список runtime](https://research.google.com/colaboratory/runtime-version-faq.html).
Торговой среды, NAV и обучения PPO нет.

Перед запуском опубликуйте актуальные файлы этапа 1 в `Go9shi/Science_Paper`.
Откройте notebook из нужной версии GitHub или загрузите этот файл в Colab.
Запускайте сверху вниз в новой сессии. Исходники клонируются в `/content`,
а результаты и snapshot сохраняются в личный Google Drive.
Train 2010–2018, validation 2019–2021, test 2022–2025; диагностики доходностей только train.

## 1. Получение кода из GitHub и PROJECT_ROOT
`GIT_REF` может быть веткой, тегом или полным commit SHA. Для первого запуска
допустим `main`; для воспроизведения используйте SHA из отчёта успешного запуска.
Каждое выполнение этой ячейки создаёт новый checkout, не перезаписывая предыдущий.
Для приватного репозитория задайте `USE_GITHUB_SECRET=True` и добавьте в Colab Secrets
секрет `GITHUB_TOKEN` с правом чтения Contents. Токен не включается в URL или артефакты.
Версия notebook должна соответствовать выбранным исходникам.

In [ ]:
import sys
import os
import re
import json
import subprocess
import tempfile
from pathlib import Path

try:
    import google.colab
except ImportError as error:
    raise RuntimeError("Запуск разрешён только в Google Colab, не в локальном runtime") from error
if not Path("/content").is_dir() or "COLAB_RELEASE_TAG" not in os.environ:
    raise RuntimeError("Нужна управляемая сессия Google Colab с каталогом /content")
if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError("Нужен Python 3.11/3.12. Выберите Runtime > Change runtime type > Runtime Version 2026.07 (Python 3.12).")

GITHUB_REPOSITORY = "Go9shi/Science_Paper"
GIT_REF = "main"  # Для повторения замените на git_commit из metadata.json.
USE_GITHUB_SECRET = False
SNAPSHOT_ID = "daily-v1"  # Тот же ID: reuse; новая выгрузка: новый ID.
NOTEBOOK_REVISION = "stage1-colab-validation-v2"
REPO_URL = f"https://github.com/{GITHUB_REPOSITORY}.git"
if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", GITHUB_REPOSITORY):
    raise ValueError("Ожидается repository в формате owner/name")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._/-]*", GIT_REF):
    raise ValueError("Некорректный GIT_REF")
if not re.fullmatch(r"[A-Za-z0-9_-]+", SNAPSHOT_ID):
    raise ValueError("Некорректный SNAPSHOT_ID")

REPO_ROOT = Path(tempfile.mkdtemp(prefix="Science_Paper-stage1-", dir="/content"))
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
with tempfile.TemporaryDirectory(prefix="stage1-auth-", dir="/content") as auth_dir:
    if USE_GITHUB_SECRET:
        from google.colab import userdata
        try:
            git_env["STAGE1_GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
        except Exception as error:
            raise RuntimeError("Разрешите доступ к Colab Secret GITHUB_TOKEN с правом чтения репозитория") from error
        if not git_env["STAGE1_GITHUB_TOKEN"]:
            raise RuntimeError("GITHUB_TOKEN пуст")
        askpass = Path(auth_dir) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['STAGE1_GITHUB_TOKEN'])\n",
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        git_env["GIT_ASKPASS"] = str(askpass)
    try:
        subprocess.run(["git", "-c", "credential.helper=", "clone", "--quiet", "--no-checkout",
                        REPO_URL, str(REPO_ROOT)], env=git_env, check=True, capture_output=True, text=True, timeout=180)
        if re.fullmatch(r"[0-9a-fA-F]{7,40}", GIT_REF):
            # Full clone already contains reachable history; resolve short SHAs locally.
            resolved = subprocess.run(
                ["git", "-C", str(REPO_ROOT), "rev-parse", "--verify", f"{GIT_REF}^{{commit}}"],
                capture_output=True, text=True,
            )
            if resolved.returncode == 0:
                SOURCE_COMMIT = resolved.stdout.strip()
            elif len(GIT_REF) < 40:
                raise ValueError("Short SHA was not found or is ambiguous. Use the full 40-character commit SHA.")
            else:
                subprocess.run(["git", "-c", "credential.helper=", "-C", str(REPO_ROOT), "fetch", "--quiet", "origin", GIT_REF],
                               env=git_env, check=True, capture_output=True, text=True, timeout=180)
                SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "FETCH_HEAD"], text=True).strip()
        else:
            subprocess.run(["git", "-c", "credential.helper=", "-C", str(REPO_ROOT), "fetch", "--quiet", "origin", GIT_REF],
                           env=git_env, check=True, capture_output=True, text=True, timeout=180)
            SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "FETCH_HEAD"], text=True).strip()
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--quiet", "--detach", SOURCE_COMMIT], check=True)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as error:
        raise RuntimeError("Не удалось получить код GitHub. Проверьте сеть, GIT_REF и доступ; для private используйте Secret.") from error
    finally:
        git_env.pop("STAGE1_GITHUB_TOKEN", None)
        git_env.pop("GIT_ASKPASS", None)

PROJECT_ROOT = REPO_ROOT / "research"
required = ["AGENTS.md", "src/__init__.py", "src/data.py", "src/volatility.py", "src/validation.py",
            "tests/test_data.py", "tests/test_volatility.py", "tests/test_validation.py",
            "requirements-colab.txt", "docs/METHODOLOGY.md", "docs/STAGE1_VALIDATION.md",
            "notebooks/01_data_preparation.ipynb"]
missing = [name for name in required if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Commit {SOURCE_COMMIT} не содержит актуальный этап 1: {missing}. "
                            "Опубликуйте файлы и выберите содержащий их ref; локальные файлы не подменяют GitHub.")
source_notebook = json.loads((PROJECT_ROOT / "notebooks/01_data_preparation.ipynb").read_text(encoding="utf-8"))
if source_notebook["metadata"].get("stage1_revision") != NOTEBOOK_REVISION:
    raise RuntimeError("Версия открытого notebook не соответствует checkout; откройте notebook из выбранного commit")
print("GitHub:", GITHUB_REPOSITORY)
print("Resolved commit:", SOURCE_COMMIT)
print("PROJECT_ROOT:", PROJECT_ROOT)

## 2. Google Drive: подключение и проба записи/чтения
Авторизация производится интерфейсом Google. Проверяется именно MyDrive,
затем запись и чтение уникального probe-файла. Probe удаляется после проверки;
исходные данные и прошлые запуски не удаляются.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    raise RuntimeError("Google Drive не подключён: каталог MyDrive недоступен")
OUTPUT_ROOT = DRIVE_ROOT / "ppo_turnover_research" / "stage1"
PREFLIGHT_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
PREFLIGHT_DIR = OUTPUT_ROOT / "preflight" / PREFLIGHT_ID
try:
    PREFLIGHT_DIR.mkdir(parents=True, exist_ok=False)
    probe = PREFLIGHT_DIR / "drive-probe.txt"
    probe_value = uuid4().hex
    probe.write_text(probe_value, encoding="utf-8")
    if probe.read_text(encoding="utf-8") != probe_value:
        raise OSError("Drive read-back mismatch")
    probe.unlink()
except OSError as error:
    raise RuntimeError(f"Нет надёжного чтения/записи Google Drive: {OUTPUT_ROOT}") from error
print("Drive read/write: OK")
print("Preflight logs:", PREFLIGHT_DIR)

## 3. Установка и проверка импортов
При несовпадении уже загруженных библиотек перезапустите сеанс Colab и выполните
ячейки сверху. Полный `pip check` сохраняется: конфликт стороннего предустановленного
пакета не скрывается, но сам по себе не доказывает ошибку данного pipeline.
Прямые версии и изолированный subprocess-import проекта обязательны.

In [ ]:
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", str(PROJECT_ROOT / "requirements-colab.txt")],
    text=True, capture_output=True,
)
(PREFLIGHT_DIR / "pip-install.log").write_text(install.stdout + "\n" + install.stderr, encoding="utf-8")
if install.returncode != 0:
    print(install.stdout[-4000:])
    print(install.stderr[-4000:])
    raise RuntimeError(f"Ошибка установки зависимостей; лог: {PREFLIGHT_DIR / 'pip-install.log'}")
print("Установка завершена; лог:", PREFLIGHT_DIR / "pip-install.log")

In [ ]:
import importlib
import importlib.metadata

requirements = {}
for line in (PROJECT_ROOT / "requirements-colab.txt").read_text(encoding="utf-8").splitlines():
    if "==" in line and not line.lstrip().startswith("#"):
        package, version = line.strip().split("==")
        requirements[package] = version
for package, version in requirements.items():
    installed = importlib.metadata.version(package)
    loaded = getattr(sys.modules.get(package), "__version__", installed)
    if installed != version or loaded != version:
        raise RuntimeError(f"{package}: installed={installed}, loaded={loaded}, expected={version}. "
                           "Перезапустите сеанс Colab и выполните notebook сверху.")

smoke_source = (
    "import importlib, json; "
    f"names = {list(requirements)!r} + ['src.data', 'src.volatility', 'src.validation']; "
    "modules = [importlib.import_module(n) for n in names]; "
    "print(json.dumps({n: str(m.__file__) for n, m in zip(names, modules)}, indent=2))"
)
project_env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT), "MPLBACKEND": "Agg"}
smoke = subprocess.run([sys.executable, "-c", smoke_source], cwd=PROJECT_ROOT,
                       env=project_env, capture_output=True, text=True)
(PREFLIGHT_DIR / "imports.log").write_text(smoke.stdout + "\n" + smoke.stderr, encoding="utf-8")
if smoke.returncode != 0:
    print(smoke.stderr[-4000:])
    raise RuntimeError(f"Ошибка импорта; см. {PREFLIGHT_DIR / 'imports.log'}; попробуйте чистую сессию Colab")
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
(PREFLIGHT_DIR / "pip-check.txt").write_text(pip_check.stdout + "\n" + pip_check.stderr, encoding="utf-8")
if pip_check.returncode != 0:
    print("ПРЕДУПРЕЖДЕНИЕ: pip check обнаружил конфликты среды; изучите pip-check.txt. Совместимость всего Colab не подтверждена.")
environment = {
    "direct_versions": {p: importlib.metadata.version(p) for p in requirements},
    "project_imports": "passed", "subprocess_module_paths": json.loads(smoke.stdout),
    "pip_check_returncode": pip_check.returncode, "drive_read_write": "passed",
    "python": sys.version, "colab_release_tag": os.environ["COLAB_RELEASE_TAG"],
}
print("Импорты проекта в новом Python-процессе: OK")

## 4. Конфигурация и архив исходников
Все импорты `src` должны разрешаться из checkout. Сохраняются commit, Git bundle,
копии исходников, хеши, runtime и preflight-логи. Не менять протокол по test.

In [ ]:
import platform
import shutil
from dataclasses import asdict

os.chdir(PROJECT_ROOT)
for name, module in list(sys.modules.items()):
    if name == "src" or name.startswith("src."):
        location = getattr(module, "__file__", None)
        if location and not Path(location).resolve().is_relative_to(PROJECT_ROOT.resolve()):
            raise RuntimeError("В памяти src из другого checkout. Перезапустите сеанс и выполните сверху.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
importlib.invalidate_caches()
from src.data import (ASSET_SECTORS, DataConfig, build_features, chronological_split,
                      compute_returns, corporate_action_audit, download_snapshot, expected_sessions,
                      sha256_file, synchronized_prices, write_json)
from src.volatility import fit_sigma_ref
from src.validation import pytest_summary, verify_saved_run

for name in ("src.data", "src.volatility", "src.validation"):
    if not Path(sys.modules[name].__file__).resolve().is_relative_to(PROJECT_ROOT.resolve()):
        raise RuntimeError(f"Импорт {name} из неверного каталога")
CONFIG = DataConfig()
CONFIG.validate()
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
SNAPSHOT_DIR = OUTPUT_ROOT / "snapshots" / SNAPSHOT_ID
source_paths = sorted(set([
    *PROJECT_ROOT.glob("src/*.py"), *PROJECT_ROOT.glob("tests/test_*.py"),
    *PROJECT_ROOT.glob(".agents/skills/*/SKILL.md"), PROJECT_ROOT / "AGENTS.md",
    PROJECT_ROOT / "requirements-colab.txt", PROJECT_ROOT / "docs/METHODOLOGY.md",
    PROJECT_ROOT / "docs/STAGE1_VALIDATION.md", PROJECT_ROOT / "notebooks/01_data_preparation.ipynb",
]))
source_hashes = {}
for original in source_paths:
    relative = original.relative_to(PROJECT_ROOT)
    target = RUN_DIR / "provenance" / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(original, target)
    source_hashes[relative.as_posix()] = sha256_file(original)
subprocess.run(["git", "-C", str(REPO_ROOT), "bundle", "create", str(RUN_DIR / "source.bundle"), "HEAD"], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "bundle", "verify", str(RUN_DIR / "source.bundle")],
               check=True, capture_output=True)
for original in PREFLIGHT_DIR.iterdir():
    if original.is_file():
        shutil.copy2(original, RUN_DIR / original.name)
write_json(RUN_DIR / "environment.json", environment)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, text=True, capture_output=True)
(RUN_DIR / "pip-freeze.txt").write_text(freeze.stdout, encoding="utf-8")
metadata = {
    "schema": 2, "run_id": RUN_ID, "status": "started",
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "execution_environment": "Google Colab", "python": sys.version, "platform": platform.platform(),
    "repository": GITHUB_REPOSITORY, "requested_ref": GIT_REF, "git_commit": SOURCE_COMMIT,
    "source_sha256": source_hashes, "config": asdict(CONFIG), "direct_dependencies": requirements,
    "asset_sectors_descriptive_not_point_in_time": ASSET_SECTORS,
    "survivorship_bias": "present: modern fixed survivor universe",
    "timing": "pre-open t features use completed bars through t-1; future execution at vendor Open[t]",
    "volatility": "sqrt(252 * mean(last 20 completed adjusted log returns squared)); uncentered",
    "sigma_ref_rule": "median sigma__SPY over training execution dates only",
    "tests": {"status": "not_run"}, "preflight_directory": str(PREFLIGHT_DIR),
}
write_json(RUN_DIR / "metadata.json", metadata)
print("Run ID:", RUN_ID)
print(json.dumps(asdict(CONFIG), indent=2))

## 5. Все тесты до получения реальных котировок
Запускается весь каталог `tests/`, включая проверку артефактов. Дополнительные
pytest-плагины среды отключены. Число тестов выводится из фактического JUnit;
ошибки, пропуски тестов или пустой отчёт не допускаются.

In [ ]:
test_env = {**project_env, "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"}
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q", f"--junitxml={RUN_DIR / 'pytest.xml'}"],
    cwd=PROJECT_ROOT, env=test_env, capture_output=True, text=True,
)
(RUN_DIR / "pytest.log").write_text(test_result.stdout + "\n" + test_result.stderr, encoding="utf-8")
print(test_result.stdout)
if test_result.returncode != 0:
    metadata["status"] = "tests_failed"
    metadata["tests"] = {"status": "failed", "exit_code": test_result.returncode}
    write_json(RUN_DIR / "metadata.json", metadata)
    raise RuntimeError(f"Тесты не прошли; загрузка не разрешена. См. {RUN_DIR / 'pytest.log'}")
try:
    summary = pytest_summary(RUN_DIR / "pytest.xml")
except Exception:
    metadata["status"] = "tests_failed"
    write_json(RUN_DIR / "metadata.json", metadata)
    raise
metadata["tests"] = {**summary, "exit_code": test_result.returncode, "log": "pytest.log", "junit": "pytest.xml"}
write_json(RUN_DIR / "metadata.json", metadata)
print("Фактически выполненные тесты:", summary)

## 6. Snapshot Yahoo Finance
Повторно используем только полный snapshot с совпадающей конфигурацией и SHA-256.
После ошибки/обрыва изучите незавершённый snapshot и задайте новый ID. Автоматических
подмен источника/тикеров нет. Snapshot — таблицы после обработки yfinance, не исходный HTTP.
Библиотека может сама сортировать и удалять дубликаты до передачи таблиц проекту.

In [ ]:
if metadata["tests"].get("status") != "passed" or metadata["tests"].get("exit_code") != 0:
    raise RuntimeError("До загрузки необходимо успешно выполнить ячейку всех тестов")
pytest_summary(RUN_DIR / "pytest.xml")
snapshot_reused = (SNAPSHOT_DIR / "snapshot.json").is_file()
try:
    raw_frames, snapshot_manifest = download_snapshot(CONFIG, SNAPSHOT_DIR)
except Exception as error:
    metadata.update(status="download_failed", error_type=type(error).__name__, error=str(error))
    write_json(RUN_DIR / "metadata.json", metadata)
    raise
metadata["snapshot"] = {
    "id": SNAPSHOT_ID, "path": str(SNAPSHOT_DIR), "reused": snapshot_reused,
    "manifest": snapshot_manifest, "manifest_sha256": sha256_file(SNAPSHOT_DIR / "snapshot.json"),
}
write_json(RUN_DIR / "metadata.json", metadata)
print("Snapshot:", SNAPSHOT_DIR)
print("Reused:", snapshot_reused, "Retrieved at:", snapshot_manifest["retrieved_at_utc"])

## 7. Подготовка, причинность и разбиение
Полная сетка NYSE; пропуски/дубликаты/невалидные бары — ошибка без fill/drop.
Признаки датированы исполнением t и используют только t−1 и раньше.
`sigma_ref` оценивается только на train. Test используется здесь исключительно
для технического контроля покрытия, без подбора окна/порогов/состава.

In [ ]:
try:
    sessions = expected_sessions(CONFIG.download_start, CONFIG.end_exclusive)
    adjusted_close, vendor_open, volume = synchronized_prices(raw_frames, sessions, CONFIG.symbols)
    simple_returns = compute_returns(adjusted_close)
    log_returns = compute_returns(adjusted_close, "log")
    features = build_features(adjusted_close, CONFIG.window, CONFIG.annualization)
    splits = chronological_split(
        features, study_start=CONFIG.study_start, validation_start=CONFIG.validation_start,
        test_start=CONFIG.test_start, end_exclusive=CONFIG.end_exclusive,
    )
    sigma_ref = fit_sigma_ref(splits["train"][f"sigma__{CONFIG.market}"],
                              train_end_exclusive=CONFIG.validation_start)
    adjustment_factor, corporate_actions, price_contract = corporate_action_audit(raw_frames)
except Exception as error:
    metadata.update(status="quality_failed", error_type=type(error).__name__, error=str(error))
    write_json(RUN_DIR / "metadata.json", metadata)
    raise
split_summary = {
    name: {"rows": len(part), "first_execution_date": str(part.index.min().date()),
           "last_execution_date": str(part.index.max().date()),
           "missing_feature_values": int(part.isna().sum().sum())}
    for name, part in splits.items()
}
metadata.update(splits=split_summary, sigma_ref=sigma_ref, price_contract=price_contract)
metadata["quality"] = {"missing_sessions": 0, "duplicate_dates_in_delivered_tables": 0,
                       "missing_bar_values": 0, "imputation": "none", "row_dropping_by_project": "none",
                       "warmup_rows": int((features.index < CONFIG.study_start).sum())}
write_json(RUN_DIR / "metadata.json", metadata)
print(json.dumps(split_summary, indent=2))
print("Train-only sigma_ref:", sigma_ref)

## 8. Контракт цен и корпоративных действий
`Adj Close` — только сигнал. `vendor Open` — отдельная база будущего исполнения.
Сохраняются `Adj Close / Close`, дивиденды, splits и события (включая Capital Gains,
если поле возвращено). Их значения за t не входят в наблюдение до open t.
Смена adjustment factor без reported event — флаг источника, не автоматический ремонт.
Этот аудит **не подтверждает** правильность будущей бухгалтерии NAV: она пока отсутствует.
Нельзя одновременно повторно применять split к уже split-adjusted ценам или дважды
учитывать дивиденды. `price_contract.json` фиксирует это ограничение.

In [ ]:
print(json.dumps(price_contract, ensure_ascii=False, indent=2))
if price_contract["unexplained_factor_change_rows"]:
    print("ПРЕДУПРЕЖДЕНИЕ: есть изменения adjustment factor без reported event. Нужен обзор corporate_actions.parquet.")
print("Корпоративные действия не являются признаками. Исполнение/NAV не валидированы.")

## 9. Диагностики train
Только качество исходных данных, не эффективность стратегии. Не фильтровать universe
по test. Крупные скачки и нулевые объёмы требуют разбора источника/событий.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

train_dates = splits["train"].index
diagnostics = pd.DataFrame(index=CONFIG.symbols)
diagnostics["train_median_vendor_dollar_volume"] = pd.Series({
    s: (raw_frames[s].loc[train_dates, "Close"] * volume.loc[train_dates, s]).median()
    for s in CONFIG.symbols
})
diagnostics["train_zero_volume_days"] = volume.loc[train_dates].eq(0).sum()
diagnostics["train_abs_daily_return_gt_25pct"] = simple_returns.loc[train_dates].abs().gt(.25).sum()
display(diagnostics)
diagnostics.to_csv(RUN_DIR / "train_diagnostics.csv", index_label="symbol")

fig, axes = plt.subplots(3, 1, figsize=(13, 13), constrained_layout=True)
train_prices = adjusted_close.loc[train_dates]
(train_prices / train_prices.iloc[0]).plot(ax=axes[0], linewidth=.8)
axes[0].set(title="Train: adjusted close / first train close (not strategy NAV)", ylabel="Relative level")
splits["train"][f"sigma__{CONFIG.market}"].plot(ax=axes[1], label="SPY pre-open sigma")
axes[1].axhline(sigma_ref, color="red", linestyle="--", label="Train median sigma_ref")
axes[1].legend()
axes[1].set(title="Train: 20 completed daily log returns, uncentered RMS", ylabel="Annualized sigma")
simple_returns.loc[train_dates, CONFIG.market].hist(ax=axes[2], bins=70)
axes[2].set(title="Train: SPY adjusted daily simple returns", xlabel="Return", ylabel="Count")
fig.savefig(RUN_DIR / "train_diagnostics.png", dpi=140)
plt.show()
plt.close(fig)

## 10. Сохранение prepared-артефактов
Таблицы признаков отделены от текущих цен и событий. Статус пока `prepared`,
не `complete`. Следующая ячейка заново читает файлы с Drive и сверяет их с snapshot.

In [ ]:
for name, digest in source_hashes.items():
    if sha256_file(PROJECT_ROOT / name) != digest:
        raise RuntimeError(f"Исходник изменился во время запуска: {name}; нужен новый запуск с фиксированным commit")
current_commit = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
if current_commit != SOURCE_COMMIT:
    raise RuntimeError("HEAD checkout изменился во время запуска")
for name, digest in snapshot_manifest["sha256"].items():
    if sha256_file(SNAPSHOT_DIR / name) != digest:
        raise RuntimeError(f"Snapshot изменился: {name}")
tables = {
    "adjusted_close": adjusted_close, "vendor_open": vendor_open, "volume": volume,
    "vendor_close": pd.DataFrame({s: raw_frames[s]["Close"] for s in CONFIG.symbols}),
    "dividends_vendor": pd.DataFrame({s: raw_frames[s]["Dividends"] for s in CONFIG.symbols}),
    "stock_splits_vendor": pd.DataFrame({s: raw_frames[s]["Stock Splits"] for s in CONFIG.symbols}),
    "adjustment_factor": adjustment_factor, "corporate_actions": corporate_actions,
    "simple_returns_close_to_close": simple_returns, "log_returns_close_to_close": log_returns,
    **{f"features_{name}": part for name, part in splits.items()},
}
for name, table in tables.items():
    table.to_parquet(RUN_DIR / f"{name}.parquet", index=True)
write_json(RUN_DIR / "price_contract.json", price_contract)
metadata["files_sha256"] = {
    p.relative_to(RUN_DIR).as_posix(): sha256_file(p)
    for p in sorted(RUN_DIR.rglob("*")) if p.is_file() and p.name not in {"metadata.json", "metadata.sha256"}
}
metadata["status"] = "prepared"
write_json(RUN_DIR / "metadata.json", metadata)
print("Prepared:", RUN_DIR, "— требуется финальная проверка")

## 11. Итоговая проверочная ячейка
Проверяет реальные JUnit cases, хеши исходников/snapshot/артефактов, чтение Parquet,
воспроизведение признаков из архивированного snapshot, границы splits и train-only
reference. Только после этого записывает `complete` и `validation_report.json`.
Отчёт не означает, что Codex уже изучил результаты, или что будущая торговая модель готова.

In [ ]:
try:
    verification = verify_saved_run(RUN_DIR)
except Exception as error:
    metadata.update(status="validation_failed", error_type=type(error).__name__, error=str(error))
    write_json(RUN_DIR / "metadata.json", metadata)
    raise
write_json(RUN_DIR / "validation_report.json", verification)
metadata["files_sha256"]["validation_report.json"] = sha256_file(RUN_DIR / "validation_report.json")
metadata["status"] = "complete"
metadata["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
write_json(RUN_DIR / "metadata.json", metadata)
(RUN_DIR / "metadata.sha256").write_text(sha256_file(RUN_DIR / "metadata.json") + "  metadata.json\n", encoding="utf-8")
if json.loads((RUN_DIR / "metadata.json").read_text(encoding="utf-8"))["status"] != "complete":
    raise RuntimeError("Финальный metadata.json не прочитан обратно")
if (RUN_DIR / "metadata.sha256").read_text(encoding="utf-8").split()[0] != sha256_file(RUN_DIR / "metadata.json"):
    raise RuntimeError("Контрольная сумма metadata.json не совпала после чтения с Drive")
print("STAGE 1: COMPLETE — подтверждён фактический запуск этого notebook в Colab")
print("Commit:", verification["source_commit"])
print("Тесты:", verification["tests"])
print("Акции:", verification["assets"], "Всего рядов со SPY:", verification["symbols"])
print("Сессии snapshot:", verification["snapshot_sessions"])
for name, item in verification["splits"].items():
    print(f"{name}: {item['rows']} строк, {item['first_execution_date']} .. {item['last_execution_date']}")
print("Run directory:", verification["run_directory"])
print("Snapshot directory:", verification["snapshot_directory"])
print("Snapshot reused:", verification["snapshot_reused"])
print("Пришлите metadata.json + metadata.sha256, validation_report.json, pytest.log, pytest.xml, environment.json,")
print("pip-check.txt, pip-freeze.txt, price_contract.json, snapshot.json и вывод этой ячейки.")
print("Обучение PPO, торговая среда и финансовая бухгалтерия исполнения не выполнялись.")